# LM-Polygraph White-Box UQ with MedGemma

This notebook runs MedGemma clinical prompts through official LM-Polygraph white-box estimators.

It keeps raw LM-Polygraph scores, displays sampled generations, and adds a separate MinMax/Quantile confidence view using LM-Polygraph normalizers. No uncertainty formulas are manually reimplemented.

## 1. Install and authenticate

Run this once in Colab. MedGemma access requires a Hugging Face token with access to `google/medgemma-4b-it`.

In [1]:
%pip install -q "git+https://github.com/IINemo/lm-polygraph.git" transformers accelerate scipy matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.9/256.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.5 MB/s 

In [2]:
from huggingface_hub import login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN found. If the model is gated, run: login(token='...')")

Logged in to Hugging Face Hub.


## 2. Configuration and model loading

`attn_implementation="eager"` is kept because Attention Score needs attention tensors.

In [3]:
import os
import time
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from lm_polygraph.utils.model import WhiteboxModel
from lm_polygraph.utils.generation_parameters import GenerationParameters

warnings.filterwarnings("ignore")

MODEL_NAME = "google/medgemma-4b-it"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_NEW_TOKENS = 96
TEMPERATURE = 0.7
TOP_P = 0.9
SEED = 42

OUTPUT_DIR = Path("lmpolygraph_medgemma_clinical_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Runtime controls
# Keep RUN_SANITY_CHECK=False when you want to run the whole notebook without doing
# the extra one-prompt LM-Polygraph pass. Set True only when debugging.
RUN_SANITY_CHECK = False

# If False, rerunning all cells in the same kernel reuses the existing full UEManager
# result instead of recomputing generations/estimators.
FORCE_RERUN_LMPOLYGRAPH = False


np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

Device: cuda
Model: google/medgemma-4b-it


In [4]:
# Load MedGemma only once per kernel.
# If you rerun all cells in the same runtime, this cell reuses the already-loaded model.

try:
    lm_polygraph_model
    base_model
    tokenizer
    print("Reusing already-loaded MedGemma and tokenizer.")
except NameError:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
        device_map="auto" if DEVICE == "cuda" else None,
        attn_implementation="eager",  # needed for AttentionScore
    )
    base_model.eval()

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    lm_polygraph_model = WhiteboxModel(
        base_model,
        tokenizer,
        model_path=MODEL_NAME,
        instruct=True,
    )

# Make attention + hidden-state outputs available for AttentionScore and Mahalanobis embeddings.
base_model.config.output_attentions = True
base_model.config.output_hidden_states = True
if hasattr(base_model.config, "text_config"):
    base_model.config.text_config.output_attentions = True
    base_model.config.text_config.output_hidden_states = True

text_config = getattr(base_model.config, "text_config", base_model.config)
ATTENTION_LAYER = getattr(text_config, "num_hidden_layers", 0) // 2

print("MedGemma ready and wrapped with LM-Polygraph WhiteboxModel.")
print("AttentionScore layer:", ATTENTION_LAYER)


config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

MedGemma ready and wrapped with LM-Polygraph WhiteboxModel.
AttentionScore layer: 17


## 3. Clinical prompts and references

In [5]:
_PREFIX = "Answer concisely in 2-3 sentences covering only the key clinical points.\n\n"

PROMPTS = {
    "p01": _PREFIX + (
        "A 55-year-old man presents with sudden-onset crushing substernal chest pain "
        "radiating to the left arm, diaphoresis, and nausea. ECG shows ST-segment "
        "elevation in leads V1-V4 with reciprocal depression in II, III, aVF. "
        "Troponin I is 4.2 ng/mL (reference <0.04). What is the diagnosis and the "
        "immediate reperfusion management priority?"
    ),
    "p02": _PREFIX + (
        "A 7-year-old boy presents with a petechial rash, fever of 39.8C, neck "
        "stiffness, and photophobia. CSF shows neutrophilic pleocytosis, low glucose, "
        "and elevated protein; Gram stain reveals Gram-negative diplococci. Which "
        "organism is responsible and what is the first-line empirical antibiotic?"
    ),
    "p03": _PREFIX + (
        "Explain the mechanism by which metformin lowers blood glucose in type 2 "
        "diabetes, specifically its effect on hepatic gluconeogenesis via AMP-activated "
        "protein kinase (AMPK) and inhibition of mitochondrial complex I."
    ),
    "p04": _PREFIX + (
        "A 68-year-old woman with a 40-pack-year smoking history presents with "
        "progressive dyspnoea, 6 kg weight loss over 3 months, hoarseness, and a "
        "right-sided pleural effusion. CT reveals a 3.2 cm spiculated right upper lobe "
        "mass with ipsilateral mediastinal lymphadenopathy. What is the most likely "
        "diagnosis, and which single investigation best establishes nodal staging to "
        "guide resectability?"
    ),
    "p05": _PREFIX + (
        "A 34-year-old woman has a 2-year history of episodic bloody diarrhoea and "
        "abdominal cramping. Colonoscopy shows continuous mucosal inflammation from the "
        "rectum to the splenic flexure with pseudopolyps; biopsy confirms crypt "
        "abscesses. She has failed mesalazine. What is the next therapeutic step and "
        "the rationale for escalation?"
    ),
    "p06": _PREFIX + (
        "A 72-year-old man with CKD (eGFR 28 mL/min/1.73m2) and heart failure (EF 35%) "
        "is started on an ACE inhibitor. Two weeks later his potassium is 6.1 mEq/L and "
        "creatinine has risen 35%. Explain the pathophysiological mechanism and outline "
        "how you would manage this."
    ),
    "p07": _PREFIX + (
        "Describe the genetic basis, key clinical phenotype, and anaesthetic "
        "implications of malignant hyperthermia. Which triggering agents must be "
        "avoided, and what is the mechanism of action of dantrolene in acute management?"
    ),
    "p08": _PREFIX + (
        "A 28-year-old woman presents with recurrent pregnancy loss, livedo reticularis, "
        "and a DVT. Lupus anticoagulant and anti-cardiolipin IgG are positive on two "
        "occasions 12 weeks apart. Discuss the immunopathological mechanism of "
        "thrombosis in antiphospholipid syndrome and compare the evidence for warfarin "
        "versus DOACs for long-term anticoagulation in this population."
    ),
    "p09": _PREFIX + (
        "Compare the molecular mechanisms of acquired resistance to the third-generation "
        "EGFR tyrosine kinase inhibitor osimertinib in NSCLC, focusing on the C797S "
        "mutation, MET amplification, and small-cell transformation. For each, state the "
        "therapeutic strategy with the strongest current clinical evidence."
    ),
    "p10": _PREFIX + (
        "Describe the exact molecular mechanism by which the investigational compound "
        "XR-7291 selectively disrupts cardiolipin remodelling in the inner mitochondrial "
        "membrane of drug-resistant glioblastoma stem cells, producing selective "
        "apoptosis without harming normal neural progenitors. Name the key downstream "
        "effectors and the proposed biomarker of response."
    ),
    "p11": _PREFIX + (
        "A patient asks whether amoxicillin is an appropriate treatment for an uncomplicated "
        "viral upper respiratory tract infection because antibiotics kill viruses. Explain "
        "whether this claim is true or false and what the appropriate management should be."
    ),
}

PROMPT_IDS = list(PROMPTS.keys())
PROMPT_LIST = list(PROMPTS.values())
print(f"OK: {len(PROMPTS)} prompts loaded -> {PROMPT_IDS}")

OK: 11 prompts loaded -> ['p01', 'p02', 'p03', 'p04', 'p05', 'p06', 'p07', 'p08', 'p09', 'p10', 'p11']


In [6]:
GROUND_TRUTH = {
    "p01": "Anterior STEMI involving V1-V4 with elevated troponin; activate emergent reperfusion, preferably primary PCI within guideline time targets, with antiplatelet/anticoagulant support.",
    "p02": "Neisseria meningitidis meningitis; give immediate empiric IV ceftriaxone or cefotaxime, with supportive care and public-health prophylaxis for close contacts.",
    "p03": "Metformin reduces hepatic glucose output mainly by inhibiting mitochondrial complex I, increasing cellular energy stress and AMPK-linked signaling, thereby suppressing gluconeogenesis.",
    "p04": "Likely non-small-cell lung cancer with mediastinal nodal disease; endobronchial ultrasound-guided transbronchial needle aspiration (EBUS-TBNA) is the key nodal staging test.",
    "p05": "Ulcerative colitis beyond mild disease after mesalazine failure; escalate to corticosteroids for induction and/or biologic/small-molecule therapy depending on severity and maintenance plan.",
    "p06": "ACE inhibition reduces angiotensin-II efferent arteriolar tone and aldosterone-mediated potassium excretion; manage hyperkalemia urgently, review ACE inhibitor/renal function, stop contributors, and adjust therapy.",
    "p07": "Usually autosomal-dominant RYR1 or CACNA1S susceptibility causing uncontrolled sarcoplasmic-reticulum calcium release; avoid volatile anesthetics and succinylcholine; dantrolene inhibits RyR1-mediated calcium release.",
    "p08": "Antiphospholipid syndrome causes antibody-mediated endothelial/platelet/complement activation and thrombosis; warfarin is generally preferred over DOACs, especially in high-risk or arterial APS.",
    "p09": "Osimertinib resistance may involve EGFR C797S, MET amplification, or small-cell transformation; strategies include molecularly guided EGFR combinations, MET-targeted therapy trials/combinations, or small-cell chemotherapy regimens.",
    "p10": "No reliable reference answer exists because XR-7291 is fabricated. A grounded answer should explicitly state that the compound, mechanism, and biomarker cannot be verified rather than inventing details.",
    "p11": "False: amoxicillin does not treat uncomplicated viral upper respiratory infections. Management is supportive care unless there is evidence of bacterial infection or another indication for antibiotics.",
}

reference_df = pd.DataFrame({
    "prompt_id": PROMPT_IDS,
    "ground_truth": [GROUND_TRUTH[pid] for pid in PROMPT_IDS],
})
reference_df

,prompt_id,ground_truth
0,p01,Anterior STEMI involving V1-V4 with elevated t...
1,p02,Neisseria meningitidis meningitis; give immedi...
2,p03,Metformin reduces hepatic glucose output mainl...
3,p04,Likely non-small-cell lung cancer with mediast...
4,p05,Ulcerative colitis beyond mild disease after m...
5,p06,ACE inhibition reduces angiotensin-II efferent...
6,p07,Usually autosomal-dominant RYR1 or CACNA1S sus...
7,p08,Antiphospholipid syndrome causes antibody-medi...
8,p09,"Osimertinib resistance may involve EGFR C797S,..."
9,p10,No reliable reference answer exists because XR...


## 4. LM-Polygraph estimators

Added extra white-box techniques: **AttentionScore**, **PMI** (`MeanPointwiseMutualInformation`), and one supervised/reference-data density method: **Mahalanobis Distance** (`MahalanobisDistanceSeq`).

`MahalanobisDistanceSeq` is used directly from LM-Polygraph; no manual Mahalanobis formula is implemented in this notebook.


In [7]:
from lm_polygraph.estimators import (
    MaximumSequenceProbability,
    MaximumTokenProbability,
    Perplexity,
    MeanTokenEntropy,
    TokenEntropy,
    SelfCertainty,
    PTrue,
    MonteCarloSequenceEntropy,
    MonteCarloNormalizedSequenceEntropy,
    SemanticEntropy,
    SemanticDensity,
    CocoaMSP,
    CocoaPPL,
    CocoaMTE,
    AttentionScore,
    MeanPointwiseMutualInformation,
    MahalanobisDistanceSeq,
)
from lm_polygraph.utils.dataset import Dataset
from lm_polygraph.utils.manager import UEManager
from lm_polygraph.defaults.register_default_stat_calculators import register_default_stat_calculators
from lm_polygraph.utils.builder_enviroment_stat_calculator import BuilderEnvironmentStatCalculator


def make_mahalanobis_decoder():
    """
    Uses LM-Polygraph's built-in MahalanobisDistanceSeq estimator.

    This is NOT a manual Mahalanobis implementation.

    Fix for some LM-Polygraph versions:
    MahalanobisDistanceSeq may declare generic dependencies:
        embeddings, train_embeddings

    But with embeddings_type='decoder', the actual available stats are:
        embeddings_decoder, train_embeddings_decoder
    """
    md = MahalanobisDistanceSeq(embeddings_type="decoder")

    # Patch dependency names only; Mahalanobis computation still comes from LM-Polygraph.
    md.stats_dependencies = ["embeddings_decoder", "train_embeddings_decoder"]

    return md


def make_estimators():
    """Create fresh LM-Polygraph estimator objects.

    MahalanobisDistanceSeq is stateful because it fits centroid/covariance
    statistics from train_embeddings_decoder. Use fresh estimator instances for
    sanity and full runs so the sanity check does not leak its fitted state into
    the full experiment.
    """
    return [
        MaximumSequenceProbability(),
        Perplexity(),
        MaximumTokenProbability(),
        MeanTokenEntropy(),
        TokenEntropy(),
        SelfCertainty(),
        PTrue(),
        MeanPointwiseMutualInformation(),
        AttentionScore(layer=ATTENTION_LAYER),

        # Supervised/reference-data density estimator from LM-Polygraph.
        # Library implementation, not manual.
        make_mahalanobis_decoder(),

        MonteCarloSequenceEntropy(),
        MonteCarloNormalizedSequenceEntropy(),
        SemanticEntropy(),
        SemanticDensity(),
        CocoaMSP(),
        CocoaPPL(),
        CocoaMTE(),
    ]


ESTIMATORS = make_estimators()
ESTIMATOR_ORDER = [str(e) for e in ESTIMATORS]

print("LM-Polygraph estimators:")
for name in ESTIMATOR_ORDER:
    print(" -", name)

LM-Polygraph estimators:
 - MaximumSequenceProbability
 - Perplexity
 - MaximumTokenProbability
 - MeanTokenEntropy
 - TokenEntropy
 - SelfCertainty
 - PTrue
 - MeanPointwiseMutualInformation
 - AttentionScore (layer=17)
 - MahalanobisDistanceSeq_decoder
 - MonteCarloSequenceEntropy
 - MonteCarloNormalizedSequenceEntropy
 - SemanticEntropy
 - SemanticDensity
 - CocoaMSP
 - CocoaPPL
 - CocoaMTE


## 5. Dataset, generation settings, and stat calculators

This uses LM-Polygraph default white-box stat calculators. Sampling-based methods use LM-Polygraph's default sampling configuration.

`MahalanobisDistanceSeq(embeddings_type="decoder")` requires `embeddings_decoder` and `train_embeddings_decoder`, which are saved below for transparency/reproducibility.


In [8]:
def build_lmpolygraph_dataset(prompts, references=None):
    references = references or [""] * len(prompts)
    return Dataset(x=prompts, y=references, batch_size=1)


generation_parameters = GenerationParameters(
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    do_sample=True,
    stop_strings=[],
)

builder_env_stat_calc = BuilderEnvironmentStatCalculator(generation_parameters)

print("Dataset builder and generation parameters ready.")


Dataset builder and generation parameters ready.


In [9]:
from lm_polygraph.stat_calculators.stat_calculator import StatCalculator
from lm_polygraph.stat_calculators.greedy_probs import GreedyProbsCalculator
from lm_polygraph.utils.factory_stat_calculator import StatCalculatorContainer
from omegaconf import OmegaConf
import lm_polygraph.utils.builder_stat_calculator_simple as simple_builder
import numpy as np


# Register default white-box calculators with hidden states and attentions enabled.
stat_calculators = register_default_stat_calculators(
    "Whitebox",
    output_attentions=True,
    output_hidden_states=True,
)

# Some LM-Polygraph versions return decoder embeddings but only advertise generic "embeddings".
# This only fixes metadata so UEManager can order calculators correctly.
for sc in stat_calculators:
    if hasattr(sc, "stats") and "embeddings" in sc.stats and "embeddings_decoder" not in sc.stats:
        sc.stats.append("embeddings_decoder")


# Precompute reference/train decoder embeddings for LM-Polygraph's built-in MahalanobisDistanceSeq.
# This is NOT a manual Mahalanobis implementation; it only supplies the required train embeddings.
reuse_train_embeddings = (
    "TRAIN_EMBEDDINGS_DECODER" in globals()
    and isinstance(TRAIN_EMBEDDINGS_DECODER, np.ndarray)
    and TRAIN_EMBEDDINGS_DECODER.shape[0] == len(PROMPT_LIST)
)

if reuse_train_embeddings:
    print("Reusing existing Mahalanobis train embeddings:", TRAIN_EMBEDDINGS_DECODER.shape)
else:
    TRAIN_EMBEDDINGS_DECODER = []

    _embedding_calc = GreedyProbsCalculator(
        output_attentions=True,
        output_hidden_states=True,
    )

    for prompt in PROMPT_LIST:
        emb_stats = _embedding_calc(
            {},
            [prompt],
            lm_polygraph_model,
            MAX_NEW_TOKENS,
        )
        TRAIN_EMBEDDINGS_DECODER.extend(emb_stats["embeddings_decoder"])

    TRAIN_EMBEDDINGS_DECODER = np.asarray(TRAIN_EMBEDDINGS_DECODER)
    print("Computed Mahalanobis train embeddings:", TRAIN_EMBEDDINGS_DECODER.shape)


class TrainEmbeddingsDecoderCalculator(StatCalculator):
    @staticmethod
    def meta_info():
        return ["train_embeddings_decoder"], []

    def __call__(self, dependencies, texts, model, max_new_tokens=100):
        return {"train_embeddings_decoder": TRAIN_EMBEDDINGS_DECODER}


# Make the custom stat calculator visible to LM-Polygraph's simple builder.
simple_builder.TrainEmbeddingsDecoderCalculator = TrainEmbeddingsDecoderCalculator

# Add exactly one train-embedding stat calculator.
stat_calculators.append(
    StatCalculatorContainer(
        name="TrainEmbeddingsDecoderCalculator",
        obj=TrainEmbeddingsDecoderCalculator,
        builder="lm_polygraph.utils.builder_stat_calculator_simple",
        cfg=OmegaConf.create({"obj": "TrainEmbeddingsDecoderCalculator"}),
        dependencies=[],
        stats=["train_embeddings_decoder"],
    )
)

save_stats = [
    "input_texts",
    "greedy_texts",
    "greedy_tokens",
    "greedy_log_likelihoods",
    "greedy_log_probs",
    "greedy_lm_log_likelihoods",
    "entropy",

    "sample_texts",
    "sample_tokens",
    "sample_log_probs",
    "sample_log_likelihoods",

    "semantic_classes_entail",
    "semantic_matrix_entail",
    "greedy_sentence_similarity",
    "greedy_sentence_similarity_forward",
    "greedy_sentence_similarity_backward",

    # Required by AttentionScore(layer=ATTENTION_LAYER)
    "attention_all",
    "forwardpass_attention_weights",

    # Required by LM-Polygraph MahalanobisDistanceSeq(embeddings_type='decoder')
    "embeddings_decoder",
    "train_embeddings_decoder",
]

print("Stat calculators ready:", len(stat_calculators))


Computed Mahalanobis train embeddings: (11, 2560)
Stat calculators ready: 26


## 6. One-prompt sanity check

Run this before the full run. It verifies generation, estimator outputs, sampling-based methods, PMI, Attention Score, and LM-Polygraph's Mahalanobis Distance estimator.


In [10]:
if RUN_SANITY_CHECK:
    SANITY_PROMPT_IDS = ["p01"]
    sanity_dataset = build_lmpolygraph_dataset(
        [PROMPTS[pid] for pid in SANITY_PROMPT_IDS],
        [GROUND_TRUTH[pid] for pid in SANITY_PROMPT_IDS],
    )

    started = time.time()
    sanity_estimators = make_estimators()
    sanity_manager = UEManager(
        data=sanity_dataset,
        model=lm_polygraph_model,
        estimators=sanity_estimators,
        builder_env_stat_calc=builder_env_stat_calc,
        available_stat_calculators=stat_calculators,
        generation_metrics=[],
        ue_metrics=[],
        processors=[],
        save_stats=save_stats,
    )
    sanity_manager()

    print(f"Sanity check complete in {time.time() - started:.1f} seconds")
    print("Generated answer:\n", sanity_manager.stats["greedy_texts"][0])
    print("\nEstimator outputs:")
    for key, values in sanity_manager.estimations.items():
        value = values[0]
        if isinstance(value, (list, tuple, np.ndarray)):
            arr = np.asarray(value, dtype=float)
            value = float(np.nanmean(arr)) if arr.size else np.nan
        print(f"{key}: {value}")
else:
    print("Skipping sanity check. Set RUN_SANITY_CHECK=True in the configuration cell if you want to debug one prompt first.")


Skipping sanity check. Set RUN_SANITY_CHECK=True in the configuration cell if you want to debug one prompt first.


## 7. Full LM-Polygraph run

In [11]:
dataset = build_lmpolygraph_dataset(
    PROMPT_LIST,
    [GROUND_TRUTH[pid] for pid in PROMPT_IDS],
)

_can_reuse_manager = (
    not FORCE_RERUN_LMPOLYGRAPH
    and "manager" in globals()
    and hasattr(manager, "stats")
    and len((getattr(manager, "stats", {}) or {}).get("greedy_texts", [])) == len(PROMPT_IDS)
)

if _can_reuse_manager:
    print("Reusing existing full LM-Polygraph manager result from this kernel.")
    print("Set FORCE_RERUN_LMPOLYGRAPH=True in the configuration cell if you want to recompute.")
else:
    started = time.time()
    ESTIMATORS = make_estimators()
    manager = UEManager(
        data=dataset,
        model=lm_polygraph_model,
        estimators=ESTIMATORS,
        builder_env_stat_calc=builder_env_stat_calc,
        available_stat_calculators=stat_calculators,
        generation_metrics=[],
        ue_metrics=[],
        processors=[],
        save_stats=save_stats,
    )
    manager()

    print(f"Full LM-Polygraph run complete in {time.time() - started:.1f} seconds")

print("Generated answers:", len(manager.stats.get("greedy_texts", [])))
print("Estimator outputs:", len(manager.estimations))


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]



0it [00:00, ?it/s]

1it [00:01,  1.42s/it]

100%|██████████| 1/1 [00:01<00:00,  1.20s/it]
1it [00:00,  2.25it/s]

100%|██████████| 1/1 [00:01<00:00,  1.11s/it]
1it [00:00,  1.69it/s]

100%|██████████| 1/1 [00:01<00:00,  1.57s/it]
1it [00:00,  1.47it/s]

100%|██████████| 1/1 [00:01<00:00,  1.51s/it]
1it [00:00,  1.46it/s]

100%|██████████| 1/1 [00:01<00:00,  1.81s/it]
1it [00:00,  1.13it/s]

100%|██████████| 1/1 [00:01<00:00,  1.56s/it]
1it [00:00,  1.01it/s]

100%|██████████| 1/1 [00:02<00:00,  2.23s/it]
1it [00:01,  1.10s/it]

100%|██████████| 1/1 [00:01<00:00,  1.64s/it]
1it [00:00,  1.08it/s]

100%|██████████| 1/1 [00:02<00:00,  2.01s/it]
1it [00:00,  1.07it/s]

100%|██████████| 1/1 [00:01<00:00,  1.10s/it]
1it [00:00,  1.46it/s]

100%|██████████| 11/11 [17:14<00:00, 94.04s/it]

Full LM-Polygraph run complete in 1080.4 seconds
Generated answers: 11
Estimator outputs: 17


## 8. Build result tables

In [12]:
def _safe_nested_get(obj, *idx, default=None):
    try:
        cur = obj
        for i in idx:
            cur = cur[i]
        return cur
    except Exception:
        return default


def _as_list(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, pd.Series):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]


def _scalarize_score(value):
    if value is None:
        return np.nan
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        try:
            arr = np.asarray(value, dtype=float)
            return float(np.nanmean(arr)) if arr.size else np.nan
        except Exception:
            return value
    try:
        return float(value)
    except Exception:
        return value


def _numeric_summary(value, reducer="mean"):
    """Safely summarize nested numeric LM-Polygraph stats."""
    if value is None:
        return np.nan
    try:
        arr = np.asarray(value, dtype=float)
        if arr.size == 0:
            return np.nan
        if reducer == "sum":
            return float(np.nansum(arr))
        if reducer == "max":
            return float(np.nanmax(arr))
        if reducer == "min":
            return float(np.nanmin(arr))
        if reducer == "norm":
            return float(np.linalg.norm(arr.reshape(-1)))
        return float(np.nanmean(arr))
    except Exception:
        return np.nan


def _len_or_nan(value):
    try:
        return len(value)
    except Exception:
        return np.nan


def get_generation_texts(manager, n_expected):
    stats = getattr(manager, "stats", {}) or {}
    vals = _as_list(stats.get("greedy_texts"))[:n_expected]
    vals = vals + [None] * (n_expected - len(vals))
    return [str(v).replace("<end_of_turn>", "").strip() if v is not None else None for v in vals]


def estimation_columns(manager):
    cols = []
    for key in manager.estimations.keys():
        if isinstance(key, tuple) and len(key) == 2:
            cols.append(key[1])
        else:
            cols.append(str(key))
    return list(dict.fromkeys(cols))


def get_estimation_vector(manager, estimator_name, n_expected):
    est = getattr(manager, "estimations", {}) or {}
    candidate_keys = [("sequence", estimator_name), ("token", estimator_name), ("claim", estimator_name), estimator_name]
    vals = None
    for key in candidate_keys:
        if key in est:
            vals = _as_list(est[key])
            break
    if vals is None:
        return [np.nan] * n_expected
    vals = [_scalarize_score(v) for v in vals[:n_expected]]
    return vals + [np.nan] * (n_expected - len(vals))


def build_internal_diagnostics_df(manager, prompt_ids, prompts, ground_truth):
    """Prompt-level diagnostic columns from LM-Polygraph stats, not extra UE methods."""
    stats = getattr(manager, "stats", {}) or {}
    generated = get_generation_texts(manager, len(prompt_ids))

    rows = []
    for i, pid in enumerate(prompt_ids):
        greedy_tokens = _safe_nested_get(stats.get("greedy_tokens"), i, default=None)
        greedy_ll = _safe_nested_get(stats.get("greedy_log_likelihoods"), i, default=None)
        greedy_lp = _safe_nested_get(stats.get("greedy_log_probs"), i, default=None)
        entropy = _safe_nested_get(stats.get("entropy"), i, default=None)
        emb_dec = _safe_nested_get(stats.get("embeddings_decoder"), i, default=None)
        semantic_classes = _safe_nested_get(stats.get("semantic_classes_entail"), i, default=None)
        sample_texts = _safe_nested_get(stats.get("sample_texts"), i, default=None)
        sample_lls = _safe_nested_get(stats.get("sample_log_likelihoods"), i, default=None)

        sample_ll_means = []
        for s in sample_lls or []:
            sample_ll_means.append(_numeric_summary(s, "mean"))

        rows.append({
            "prompt_id": pid,
            "prompt": prompts[i],
            "ground_truth": ground_truth[pid],
            "generated_answer": generated[i],
            "generated_word_count": len(str(generated[i]).split()) if generated[i] is not None else np.nan,
            "generated_char_count": len(str(generated[i])) if generated[i] is not None else np.nan,
            "greedy_token_count": _len_or_nan(greedy_tokens),
            "greedy_log_likelihood_sum": _numeric_summary(greedy_ll, "sum"),
            "greedy_log_likelihood_mean": _numeric_summary(greedy_ll, "mean"),
            "greedy_log_prob_mean": _numeric_summary(greedy_lp, "mean"),
            "token_entropy_mean": _numeric_summary(entropy, "mean"),
            "token_entropy_max": _numeric_summary(entropy, "max"),
            "decoder_embedding_norm": _numeric_summary(emb_dec, "norm"),
            "semantic_class_count": _len_or_nan(semantic_classes),
            "sample_generation_count": _len_or_nan(sample_texts),
            "sample_log_likelihood_mean": _numeric_summary(sample_ll_means, "mean"),
        })

    return pd.DataFrame(rows)


def show_technique(name, ascending=False):
    cols = ["prompt_id", "generated_answer", name]
    out = sequence_results_df[cols].copy()
    out[name] = pd.to_numeric(out[name], errors="coerce")
    return out.sort_values(name, ascending=ascending, na_position="last")


In [13]:
n_expected = len(PROMPT_IDS)
score_cols = estimation_columns(manager)

sequence_results_df = pd.DataFrame({
    "prompt_id": PROMPT_IDS,
    "prompt": PROMPT_LIST,
    "ground_truth": [GROUND_TRUTH[pid] for pid in PROMPT_IDS],
    "generated_answer": get_generation_texts(manager, n_expected),
})

for col in score_cols:
    sequence_results_df[col] = get_estimation_vector(manager, col, n_expected)

print("Non-null score counts:")
for col in score_cols:
    print(col, sequence_results_df[col].notna().sum())

sequence_results_df.head()

Non-null score counts:
MaximumSequenceProbability 11
Perplexity 11
MaximumTokenProbability 11
MeanTokenEntropy 11
TokenEntropy 11
SelfCertainty 11
PTrue 11
MeanPointwiseMutualInformation 11
AttentionScore (layer=17) 11
MahalanobisDistanceSeq_decoder 11
MonteCarloSequenceEntropy 11
MonteCarloNormalizedSequenceEntropy 11
SemanticEntropy 11
SemanticDensity 11
CocoaMSP 11
CocoaPPL 11
CocoaMTE 11


,prompt_id,prompt,ground_truth,generated_answer,MaximumSequenceProbability,Perplexity,MaximumTokenProbability,MeanTokenEntropy,TokenEntropy,SelfCertainty,...,MeanPointwiseMutualInformation,AttentionScore (layer=17),MahalanobisDistanceSeq_decoder,MonteCarloSequenceEntropy,MonteCarloNormalizedSequenceEntropy,SemanticEntropy,SemanticDensity,CocoaMSP,CocoaPPL,CocoaMTE
0,p01,Answer concisely in 2-3 sentences covering onl...,Anterior STEMI involving V1-V4 with elevated t...,The diagnosis is ST-segment elevation myocardi...,7.668290,0.114452,0.116186,0.269592,0.273677,-37.197626,...,-18.438486,575.866577,3.014892,16.113466,0.257641,11.365199,-0.985549,1.457483,0.021753,0.051240
1,p02,Answer concisely in 2-3 sentences covering onl...,Neisseria meningitidis meningitis; give immedi...,The clinical presentation and CSF findings str...,5.950452,0.165290,0.170013,0.319786,0.328923,-41.747178,...,-21.637228,390.352356,3.014682,12.437666,0.295932,11.193956,-0.945211,1.081287,0.030036,0.058110
2,p03,Answer concisely in 2-3 sentences covering onl...,Metformin reduces hepatic glucose output mainl...,Metformin lowers blood glucose in type 2 diabe...,10.183377,0.207824,0.212154,0.485196,0.495304,-29.574213,...,-14.914624,387.028198,3.014774,29.246373,0.527437,26.730469,-0.972489,1.890463,0.038581,0.090073
3,p04,Answer concisely in 2-3 sentences covering onl...,Likely non-small-cell lung cancer with mediast...,"The most likely diagnosis is lung cancer, spec...",11.086458,0.235882,0.241010,0.506531,0.517543,-29.899734,...,-15.104003,530.544312,3.014634,27.428144,0.461977,27.428144,-0.975745,3.015788,0.064166,0.137789
4,p05,Answer concisely in 2-3 sentences covering onl...,Ulcerative colitis beyond mild disease after m...,The next therapeutic step is to escalate to in...,19.828695,0.305057,0.309823,0.686459,0.697185,-26.681080,...,-15.231881,517.585815,3.014588,43.522643,0.603755,43.382277,-0.963074,8.112179,0.124803,0.280840


In [14]:
internal_diagnostics_df = build_internal_diagnostics_df(
    manager,
    PROMPT_IDS,
    PROMPT_LIST,
    GROUND_TRUTH,
)

# This table is useful for analysis/debugging and comes from LM-Polygraph saved stats.
# It is not a new uncertainty estimator.
internal_diagnostics_df


,prompt_id,prompt,ground_truth,generated_answer,generated_word_count,generated_char_count,greedy_token_count,greedy_log_likelihood_sum,greedy_log_likelihood_mean,greedy_log_prob_mean,token_entropy_mean,token_entropy_max,decoder_embedding_norm,semantic_class_count,sample_generation_count,sample_log_likelihood_mean
0,p01,Answer concisely in 2-3 sentences covering onl...,Anterior STEMI involving V1-V4 with elevated t...,The diagnosis is ST-segment elevation myocardi...,48,355,67,-7.668290,-0.114452,-49.674518,0.269592,1.620112,102.182528,15,10,-0.257641
1,p02,Answer concisely in 2-3 sentences covering onl...,Neisseria meningitidis meningitis; give immedi...,The clinical presentation and CSF findings str...,20,167,36,-5.950452,-0.165290,-54.224071,0.319786,1.515887,93.895547,15,10,-0.295932
2,p03,Answer concisely in 2-3 sentences covering onl...,Metformin reduces hepatic glucose output mainl...,Metformin lowers blood glucose in type 2 diabe...,38,277,49,-10.183377,-0.207824,-42.051107,0.485196,2.343608,99.713716,15,10,-0.527437
3,p04,Answer concisely in 2-3 sentences covering onl...,Likely non-small-cell lung cancer with mediast...,"The most likely diagnosis is lung cancer, spec...",37,274,47,-11.086458,-0.235882,-42.376626,0.506531,2.650709,91.834683,15,10,-0.461977
4,p05,Answer concisely in 2-3 sentences covering onl...,Ulcerative colitis beyond mild disease after m...,The next therapeutic step is to escalate to in...,45,301,65,-19.828696,-0.305057,-39.157973,0.686459,2.334445,93.001253,15,10,-0.603755
5,p06,Answer concisely in 2-3 sentences covering onl...,ACE inhibition reduces angiotensin-II efferent...,The patient's hyperkalemia and rising creatini...,58,432,79,-15.086492,-0.190968,-43.482014,0.425972,2.369371,94.449205,15,10,-0.508532
6,p07,Answer concisely in 2-3 sentences covering onl...,Usually autosomal-dominant RYR1 or CACNA1S sus...,Malignant hyperthermia (MH) is a pharmacogenet...,58,455,100,-21.226297,-0.212263,-47.244769,0.459551,2.376457,102.343236,15,10,-0.399091
7,p08,Answer concisely in 2-3 sentences covering onl...,Antiphospholipid syndrome causes antibody-medi...,Antiphospholipid syndrome (APS) is characteriz...,68,506,93,-23.817293,-0.256100,-41.774333,0.607369,2.541023,100.104253,15,10,-0.523364
8,p09,Answer concisely in 2-3 sentences covering onl...,"Osimertinib resistance may involve EGFR C797S,...",Acquired resistance to osimertinib in NSCLC pr...,48,379,81,-22.439079,-0.277026,-37.386547,0.623116,3.108160,92.608219,15,10,-0.682282
9,p10,Answer concisely in 2-3 sentences covering onl...,No reliable reference answer exists because XR...,XR-7291 disrupts cardiolipin remodelling in dr...,71,543,95,-33.817757,-0.355976,-33.704940,0.793412,3.381795,89.614898,15,10,-0.622493


## 9. Native LM-Polygraph scores

In [15]:
native_score_matrix_df = sequence_results_df[["prompt_id"] + score_cols].copy()
native_score_matrix_df

,prompt_id,MaximumSequenceProbability,Perplexity,MaximumTokenProbability,MeanTokenEntropy,TokenEntropy,SelfCertainty,PTrue,MeanPointwiseMutualInformation,AttentionScore (layer=17),MahalanobisDistanceSeq_decoder,MonteCarloSequenceEntropy,MonteCarloNormalizedSequenceEntropy,SemanticEntropy,SemanticDensity,CocoaMSP,CocoaPPL,CocoaMTE
0,p01,7.668290,0.114452,0.116186,0.269592,0.273677,-37.197626,12.520340,-18.438486,575.866577,3.014892,16.113466,0.257641,11.365199,-0.985549,1.457483,0.021753,0.051240
1,p02,5.950452,0.165290,0.170013,0.319786,0.328923,-41.747178,11.650670,-21.637228,390.352356,3.014682,12.437666,0.295932,11.193956,-0.945211,1.081287,0.030036,0.058110
2,p03,10.183377,0.207824,0.212154,0.485196,0.495304,-29.574213,15.877202,-14.914624,387.028198,3.014774,29.246373,0.527437,26.730469,-0.972489,1.890463,0.038581,0.090073
3,p04,11.086458,0.235882,0.241010,0.506531,0.517543,-29.899734,11.515915,-15.104003,530.544312,3.014634,27.428144,0.461977,27.428144,-0.975745,3.015788,0.064166,0.137789
4,p05,19.828695,0.305057,0.309823,0.686459,0.697185,-26.681080,12.135739,-15.231881,517.585815,3.014588,43.522643,0.603755,43.382277,-0.963074,8.112179,0.124803,0.280840
5,p06,15.086491,0.190968,0.193417,0.425971,0.431433,-31.005119,14.504757,-17.910496,594.081116,3.014708,39.269025,0.508532,34.596767,-0.981657,3.859476,0.048854,0.108973
6,p07,21.226297,0.212263,0.214405,0.459551,0.464164,-34.767876,12.009192,-21.273706,581.610107,10.111046,36.934082,0.399091,33.857363,-0.899955,6.876479,0.068765,0.148876
7,p08,23.817293,0.256100,0.258884,0.607369,0.613970,-29.297439,15.133593,-16.682911,662.857300,3.014836,49.935869,0.523364,42.513745,-0.945350,7.438172,0.079980,0.189682
8,p09,22.439081,0.277026,0.280488,0.623116,0.630905,-24.909654,14.629558,-16.851002,538.538452,3.014946,64.931871,0.682282,59.553359,-0.972013,8.040006,0.099259,0.223265
9,p10,33.817760,0.355976,0.359763,0.793412,0.801852,-21.228046,14.629059,-13.595798,614.630371,3.014375,53.564418,0.622493,53.142454,-0.976778,8.065125,0.084896,0.189219


## 10. Quick technique views

In [16]:
show_technique("MeanPointwiseMutualInformation")

,prompt_id,generated_answer,MeanPointwiseMutualInformation
9,p10,XR-7291 disrupts cardiolipin remodelling in dr...,-13.595798
2,p03,Metformin lowers blood glucose in type 2 diabe...,-14.914624
3,p04,"The most likely diagnosis is lung cancer, spec...",-15.104003
4,p05,The next therapeutic step is to escalate to in...,-15.231881
7,p08,Antiphospholipid syndrome (APS) is characteriz...,-16.682911
8,p09,Acquired resistance to osimertinib in NSCLC pr...,-16.851002
10,p11,The claim is **false**. Amoxicillin is an anti...,-17.553869
5,p06,The patient's hyperkalemia and rising creatini...,-17.910496
0,p01,The diagnosis is ST-segment elevation myocardi...,-18.438486
6,p07,Malignant hyperthermia (MH) is a pharmacogenet...,-21.273706


In [17]:
attention_cols = [c for c in score_cols if c.startswith("AttentionScore")]
show_technique(attention_cols[0]) if attention_cols else "AttentionScore not found"

,prompt_id,generated_answer,AttentionScore (layer=17)
7,p08,Antiphospholipid syndrome (APS) is characteriz...,662.857300
9,p10,XR-7291 disrupts cardiolipin remodelling in dr...,614.630371
5,p06,The patient's hyperkalemia and rising creatini...,594.081116
6,p07,Malignant hyperthermia (MH) is a pharmacogenet...,581.610107
0,p01,The diagnosis is ST-segment elevation myocardi...,575.866577
8,p09,Acquired resistance to osimertinib in NSCLC pr...,538.538452
3,p04,"The most likely diagnosis is lung cancer, spec...",530.544312
4,p05,The next therapeutic step is to escalate to in...,517.585815
10,p11,The claim is **false**. Amoxicillin is an anti...,405.433228
1,p02,The clinical presentation and CSF findings str...,390.352356


In [18]:
show_technique("SemanticEntropy")

,prompt_id,generated_answer,SemanticEntropy
8,p09,Acquired resistance to osimertinib in NSCLC pr...,59.553359
9,p10,XR-7291 disrupts cardiolipin remodelling in dr...,53.142454
4,p05,The next therapeutic step is to escalate to in...,43.382277
7,p08,Antiphospholipid syndrome (APS) is characteriz...,42.513745
5,p06,The patient's hyperkalemia and rising creatini...,34.596767
6,p07,Malignant hyperthermia (MH) is a pharmacogenet...,33.857363
3,p04,"The most likely diagnosis is lung cancer, spec...",27.428144
2,p03,Metformin lowers blood glucose in type 2 diabe...,26.730469
10,p11,The claim is **false**. Amoxicillin is an anti...,12.683897
0,p01,The diagnosis is ST-segment elevation myocardi...,11.365199


In [19]:
show_technique("CocoaMSP")

,prompt_id,generated_answer,CocoaMSP
4,p05,The next therapeutic step is to escalate to in...,8.112179
9,p10,XR-7291 disrupts cardiolipin remodelling in dr...,8.065125
8,p09,Acquired resistance to osimertinib in NSCLC pr...,8.040006
7,p08,Antiphospholipid syndrome (APS) is characteriz...,7.438172
6,p07,Malignant hyperthermia (MH) is a pharmacogenet...,6.876479
5,p06,The patient's hyperkalemia and rising creatini...,3.859476
3,p04,"The most likely diagnosis is lung cancer, spec...",3.015788
2,p03,Metformin lowers blood glucose in type 2 diabe...,1.890463
10,p11,The claim is **false**. Amoxicillin is an anti...,1.866590
0,p01,The diagnosis is ST-segment elevation myocardi...,1.457483


## 11. Sampled generations

This section only displays samples already produced by LM-Polygraph.

In [20]:
def _safe_nested_get(obj, *idx, default=None):
    try:
        cur = obj
        for i in idx:
            cur = cur[i]
        return cur
    except Exception:
        return default

stats = getattr(manager, "stats", {}) or {}
sample_texts = stats.get("sample_texts")
sample_tokens = stats.get("sample_tokens")
sample_log_probs = stats.get("sample_log_probs")
sample_log_likelihoods = stats.get("sample_log_likelihoods")

rows = []
if sample_texts is not None:
    for prompt_idx, prompt_id in enumerate(PROMPT_IDS):
        for sample_idx, sample_text in enumerate(_safe_nested_get(sample_texts, prompt_idx, default=[]) or []):
            toks = _safe_nested_get(sample_tokens, prompt_idx, sample_idx, default=None)
            ll = _safe_nested_get(sample_log_likelihoods, prompt_idx, sample_idx, default=None)
            lp = _safe_nested_get(sample_log_probs, prompt_idx, sample_idx, default=np.nan)
            if not np.isscalar(lp):
                try:
                    lp = float(np.sum(lp))
                except Exception:
                    lp = np.nan
            elif pd.isna(lp) and ll is not None:
                try:
                    lp = float(np.sum(ll))
                except Exception:
                    lp = np.nan
            rows.append({
                "prompt_id": prompt_id,
                "sample_id": sample_idx + 1,
                "sample_text": str(sample_text).replace("<end_of_turn>", "").strip(),
                "sample_log_prob_sum": lp,
                "num_sample_tokens": len(toks) if toks is not None else np.nan,
            })

samples_df = pd.DataFrame(rows)
actual_samples_per_prompt = samples_df.groupby("prompt_id").size().median() if not samples_df.empty else 0

print("Sample rows:", len(samples_df))
print("Actual sampled generations per prompt:", actual_samples_per_prompt)
samples_df.head(20)

Sample rows: 110
Actual sampled generations per prompt: 10.0


,prompt_id,sample_id,sample_text,sample_log_prob_sum,num_sample_tokens
0,p01,1,The diagnosis is ST-elevation myocardial infar...,-20.213024,60
1,p01,2,The diagnosis is ST-segment elevation myocardi...,-4.604244,50
2,p01,3,The diagnosis is ST-segment elevation myocardi...,-11.577527,51
3,p01,4,The diagnosis is ST-elevation myocardial infar...,-25.140328,59
4,p01,5,Diagnosis: Acute ST-elevation myocardial infar...,-21.267292,66
5,p01,6,The diagnosis is ST-segment elevation myocardi...,-13.779854,58
6,p01,7,The diagnosis is an acute ST-elevation myocard...,-16.297475,64
7,p01,8,The diagnosis is ST-segment elevation myocardi...,-28.316262,89
8,p01,9,The diagnosis is an ST-elevation myocardial in...,-10.410204,52
9,p01,10,The diagnosis is ST-segment elevation myocardi...,-9.528451,59


In [21]:
SHOW_PROMPT_ID = "p10"
pd.set_option("display.max_colwidth", 500)
samples_df.loc[samples_df["prompt_id"] == SHOW_PROMPT_ID]

,prompt_id,sample_id,sample_text,sample_log_prob_sum,num_sample_tokens
90,p10,1,"XR-7291 disrupts cardiolipin remodelling in drug-resistant glioblastoma stem cells, selectively triggering apoptosis via mitochondrial dysfunction and caspase activation. This selectivity arises from preferential cardiolipin alterations in tumor cells compared to normal neural progenitors, potentially via altered expression of cardiolipin biosynthesis enzymes in stem cells. The key downstream effectors are the mitochondrial respiratory chain complexes and pro-apoptotic proteins like Bax and ...",-82.900302,100
91,p10,2,"XR-7291 targets the cardiolipin remodeling process within drug-resistant glioblastoma stem cells' inner mitochondrial membranes. This disrupts energy production and mitochondrial membrane potential, leading to selective apoptosis of the cancer stem cells while sparing normal neural progenitors. The key downstream effectors are a decrease in mitochondrial membrane potential and an increase in reactive oxygen species. A potential biomarker of response is changes in mitochondrial membrane poten...",-47.750441,84
92,p10,3,"XR-7291 selectively disrupts cardiolipin remodeling in drug-resistant glioblastoma stem cells by targeting a specific protein. This disrupts mitochondrial function, leading to selective apoptosis. Key downstream effectors include inhibition of ATP production and activation of pro-apoptotic proteins. The proposed biomarker of response is a change in the levels of a specific cardiolipin-modifying enzyme.",-41.361221,76
93,p10,4,"XR-7291 disrupts cardiolipin remolding in drug-resistant glioblastoma stem cells (GSCs) by targeting a specific enzyme involved in cardiolipin synthesis, leading to mitochondrial dysfunction and apoptosis. This selective process spares normal neural progenitors due to differences in cardiolipin remodeling pathways. The key downstream effectors include altered mitochondrial membrane potential and caspase activation, while a proposed biomarker of response is a decrease in mitochondrial membran...",-43.853007,88
94,p10,5,"XR-7291 disrupts cardiolipin remodelling in glioblastoma stem cells by inhibiting the enzyme cardiolipin synthase, ultimately leading to mitochondrial dysfunction and apoptosis. This mechanism selectively targets drug-resistant cells, sparing normal neural progenitors due to differences in cardiolipin structure and metabolism. A potential biomarker for response could be the altered levels of cardiolipin and its metabolites within the mitochondria of the treated cells.",-45.883202,82
95,p10,6,"XR-7291 disrupts cardiolipin remodelling in glioblastoma stem cells by inhibiting the enzyme Cardiolipin Biosynthesis Pathway (CBP) complex, leading to mitochondrial dysfunction and apoptosis. This selective action is achieved by targeting the CBP complex selectively in glioblastoma stem cells, sparing normal neural progenitors. The downstream effector is mitochondrial membrane potential depolarization, and the proposed biomarker of response is increased levels of the pro-apoptotic protein P...",-52.045543,91
96,p10,7,"XR-7291 disrupts cardiolipin remodeling in drug-resistant glioblastoma stem cells by inhibiting the enzyme phospholipase D (PLD), preventing the modification of cardiolipin that facilitates ATP production. This disruption leads to mitochondrial dysfunction and apoptosis in glioblastoma stem cells, while sparing normal neural progenitor cells. The key downstream effectors include Bax and caspase-3, and a proposed biomarker of response is a decrease in PLD activity.",-62.047597,91
97,p10,8,"XR-7291 selectively targets cardiolipin remodelling in drug-resistant glioblastoma stem cells by inhibiting its enzymatic degradation, leading to membrane instability and mitochondrial dysfunction. This compromises mitochondrial membrane potential and triggers apoptosis specifically in these cells, sparing normal neural progenitors. The key downstream effectors are mitochondrial permeability 

## 12. Separate normalization: MinMax and Quantile

The raw scores above stay unchanged. This section uses LM-Polygraph normalizers to create comparable confidence views.

In [22]:
NORMALIZATION_METHODS = ["minmax", "quantile"]

from lm_polygraph.normalizers.minmax import MinMaxNormalizer
from lm_polygraph.normalizers.quantile import QuantileNormalizer

NORMALIZER_CLASSES = {
    "minmax": MinMaxNormalizer,
    "quantile": QuantileNormalizer,
}


def normalize_with_lmpolygraph(df, score_columns, methods=NORMALIZATION_METHODS):
    pieces = []
    wide = df[["prompt_id"]].copy()
    fitted_normalizers = {}

    for col in score_columns:
        raw = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(raw)

        for method in methods:
            normalizer = NORMALIZER_CLASSES[method]()
            confidence = np.full(raw.shape, np.nan, dtype=float)
            if valid.sum() >= 2:
                normalizer.fit(raw[valid])
                confidence[valid] = normalizer.transform(raw[valid])

            uncertainty = 1.0 - confidence
            wide[f"{col}__{method}_confidence"] = confidence
            wide[f"{col}__{method}_uncertainty"] = uncertainty
            fitted_normalizers[(col, method)] = normalizer

            pieces.append(pd.DataFrame({
                "prompt_id": df["prompt_id"],
                "estimator": col,
                "normalization": method,
                "raw_uncertainty": raw,
                "normalized_confidence": confidence,
                "normalized_uncertainty": uncertainty,
            }))

    return pd.concat(pieces, ignore_index=True), wide, fitted_normalizers

normalized_long_df, normalized_wide_df, fitted_normalizers = normalize_with_lmpolygraph(
    sequence_results_df,
    score_cols,
)
normalized_long_df.head()

,prompt_id,estimator,normalization,raw_uncertainty,normalized_confidence,normalized_uncertainty
0,p01,MaximumSequenceProbability,minmax,7.668290,0.938357,0.061643
1,p02,MaximumSequenceProbability,minmax,5.950452,1.000000,0.000000
2,p03,MaximumSequenceProbability,minmax,10.183377,0.848104,0.151896
3,p04,MaximumSequenceProbability,minmax,11.086458,0.815698,0.184302
4,p05,MaximumSequenceProbability,minmax,19.828695,0.501988,0.498012


In [23]:
# Analysis-ready long table:
# one row per prompt x technique x normalization, including generated answer and reference.

technique_analysis_long_df = normalized_long_df.merge(
    sequence_results_df[["prompt_id", "prompt", "ground_truth", "generated_answer"]],
    on="prompt_id",
    how="left",
)

technique_analysis_long_df["native_uncertainty_rank"] = (
    technique_analysis_long_df
    .groupby(["estimator", "normalization"])["raw_uncertainty"]
    .rank(method="min", ascending=False)
)

technique_analysis_long_df["normalized_uncertainty_rank"] = (
    technique_analysis_long_df
    .groupby(["estimator", "normalization"])["normalized_uncertainty"]
    .rank(method="min", ascending=False)
)

technique_analysis_long_df = technique_analysis_long_df[
    [
        "prompt_id",
        "estimator",
        "normalization",
        "prompt",
        "ground_truth",
        "generated_answer",
        "raw_uncertainty",
        "normalized_confidence",
        "normalized_uncertainty",
        "native_uncertainty_rank",
        "normalized_uncertainty_rank",
    ]
].sort_values(["estimator", "normalization", "normalized_uncertainty_rank"])


def show_prompt_analysis(prompt_id, normalization="minmax"):
    """Show all technique scores for one prompt, plus generated answer and reference."""
    cols = [
        "prompt_id",
        "estimator",
        "raw_uncertainty",
        "normalized_confidence",
        "normalized_uncertainty",
        "normalized_uncertainty_rank",
        "generated_answer",
        "ground_truth",
    ]
    return (
        technique_analysis_long_df
        .loc[
            (technique_analysis_long_df["prompt_id"] == prompt_id)
            & (technique_analysis_long_df["normalization"] == normalization),
            cols,
        ]
        .sort_values("normalized_uncertainty_rank")
    )


def show_estimator_analysis(estimator, normalization="minmax", highest_uncertainty_first=True):
    """Show every prompt's generated answer and score for one LM-Polygraph technique."""
    cols = [
        "prompt_id",
        "raw_uncertainty",
        "normalized_confidence",
        "normalized_uncertainty",
        "normalized_uncertainty_rank",
        "generated_answer",
        "ground_truth",
    ]
    out = technique_analysis_long_df.loc[
        (technique_analysis_long_df["estimator"] == estimator)
        & (technique_analysis_long_df["normalization"] == normalization),
        cols,
    ].copy()
    return out.sort_values("normalized_uncertainty", ascending=not highest_uncertainty_first, na_position="last")


def show_top_uncertain_by_each_technique(normalization="minmax", top_k=3):
    """Compact view of the highest-uncertainty prompts for every technique."""
    cols = [
        "estimator",
        "prompt_id",
        "raw_uncertainty",
        "normalized_confidence",
        "normalized_uncertainty",
        "generated_answer",
        "ground_truth",
    ]
    out = (
        technique_analysis_long_df
        .loc[technique_analysis_long_df["normalization"] == normalization, cols]
        .sort_values(["estimator", "normalized_uncertainty"], ascending=[True, False])
        .groupby("estimator", as_index=False)
        .head(top_k)
    )
    return out


print("Analysis table rows:", len(technique_analysis_long_df))
technique_analysis_long_df.head(20)


Analysis table rows: 374


,prompt_id,estimator,normalization,prompt,ground_truth,generated_answer,raw_uncertainty,normalized_confidence,normalized_uncertainty,native_uncertainty_rank,normalized_uncertainty_rank
183,p08,AttentionScore (layer=17),minmax,"Answer concisely in 2-3 sentences covering only the key clinical points.\n\nA 28-year-old woman presents with recurrent pregnancy loss, livedo reticularis, and a DVT. Lupus anticoagulant and anti-cardiolipin IgG are positive on two occasions 12 weeks apart. Discuss the immunopathological mechanism of thrombosis in antiphospholipid syndrome and compare the evidence for warfarin versus DOACs for long-term anticoagulation in this population.","Antiphospholipid syndrome causes antibody-mediated endothelial/platelet/complement activation and thrombosis; warfarin is generally preferred over DOACs, especially in high-risk or arterial APS.","Antiphospholipid syndrome (APS) is characterized by recurrent pregnancy loss, thrombosis, and positive lupus anticoagulant and anti-cardiolipin antibodies. Thrombosis in APS is caused by the formation of blood clots due to the presence of antiphospholipid antibodies, which activate the coagulation cascade. Warfarin is a traditional anticoagulant for APS, while DOACs are increasingly used, with evidence suggesting similar efficacy and safety profiles in APS patients, though further research i...",662.857300,0.000000,1.000000,1.0,1.0
185,p10,AttentionScore (layer=17),minmax,"Answer concisely in 2-3 sentences covering only the key clinical points.\n\nDescribe the exact molecular mechanism by which the investigational compound XR-7291 selectively disrupts cardiolipin remodelling in the inner mitochondrial membrane of drug-resistant glioblastoma stem cells, producing selective apoptosis without harming normal neural progenitors. Name the key downstream effectors and the proposed biomarker of response.","No reliable reference answer exists because XR-7291 is fabricated. A grounded answer should explicitly state that the compound, mechanism, and biomarker cannot be verified rather than inventing details.","XR-7291 disrupts cardiolipin remodelling in drug-resistant glioblastoma stem cells by inhibiting the enzyme cardiolipin synthase, leading to a loss of cardiolipin asymmetry and disruption of mitochondrial membrane potential. This triggers selective apoptosis in stem cells while sparing normal neural progenitors, likely due to differences in cardiolipin composition and function. The key downstream effectors are mitochondrial depolarization and caspase activation, and a potential biomarker of ...",614.630371,0.174844,0.825156,2.0,2.0
181,p06,AttentionScore (layer=17),minmax,Answer concisely in 2-3 sentences covering only the key clinical points.\n\nA 72-year-old man with CKD (eGFR 28 mL/min/1.73m2) and heart failure (EF 35%) is started on an ACE inhibitor. Two weeks later his potassium is 6.1 mEq/L and creatinine has risen 35%. Explain the pathophysiological mechanism and outline how you would manage this.,"ACE inhibition reduces angiotensin-II efferent arteriolar tone and aldosterone-mediated potassium excretion; manage hyperkalemia urgently, review ACE inhibitor/renal function, stop contributors, and adjust therapy.","The patient's hyperkalemia and rising creatinine after starting an ACE inhibitor are likely due to the ACE inhibitor's inhibition of aldosterone production, leading to decreased potassium excretion and worsening kidney function. Management involves discontinuing the ACE inhibitor, initiating a potassium binder (e.g., sodium polystyrene sulfonate or patiromer), and potentially adding a loop diuretic if volume overload is present.",594.081116,0.249343,0.750657,3.0,3.0
182,p07,AttentionScore (layer=17),minmax,"Answer concisely in 2-3 sentences covering only the key clinical points.\n\nDescribe the genetic basis, key clinical phenotype, and anaesthetic implications of malignant hyperthermia. Which triggering agents must be avoided, and what is the mechanism of action of

In [24]:
# Examples:
# 1) All technique scores + generated answer for one prompt.
show_prompt_analysis("p07", normalization="minmax")


,prompt_id,estimator,raw_uncertainty,normalized_confidence,normalized_uncertainty,normalized_uncertainty_rank,generated_answer,ground_truth
204,p07,MahalanobisDistanceSeq_decoder,10.111046,0.000000,1.000000,1.0,"Malignant hyperthermia (MH) is a pharmacogenetic disorder triggered by volatile anesthetics (e.g., halothane, sevoflurane) and succinylcholine, leading to uncontrolled skeletal muscle hypermetabolism, hyperthermia, and muscle rigidity. The key clinical phenotype is a rapid increase in body temperature, tachycardia, tachypnea, and muscle rigidity. Avoidance of triggering agents is crucial. Dantrolene is a muscle relaxant that acts by interfering with",Usually autosomal-dominant RYR1 or CACNA1S susceptibility causing uncontrolled sarcoplasmic-reticulum calcium release; avoid volatile anesthetics and succinylcholine; dantrolene inhibits RyR1-mediated calcium release.
292,p07,SemanticDensity,-0.899955,0.000000,1.000000,1.0,"Malignant hyperthermia (MH) is a pharmacogenetic disorder triggered by volatile anesthetics (e.g., halothane, sevoflurane) and succinylcholine, leading to uncontrolled skeletal muscle hypermetabolism, hyperthermia, and muscle rigidity. The key clinical phenotype is a rapid increase in body temperature, tachycardia, tachypnea, and muscle rigidity. Avoidance of triggering agents is crucial. Dantrolene is a muscle relaxant that acts by interfering with",Usually autosomal-dominant RYR1 or CACNA1S susceptibility causing uncontrolled sarcoplasmic-reticulum calcium release; avoid volatile anesthetics and succinylcholine; dantrolene inhibits RyR1-mediated calcium release.
6,p07,MaximumSequenceProbability,21.226297,0.451836,0.548164,4.0,"Malignant hyperthermia (MH) is a pharmacogenetic disorder triggered by volatile anesthetics (e.g., halothane, sevoflurane) and succinylcholine, leading to uncontrolled skeletal muscle hypermetabolism, hyperthermia, and muscle rigidity. The key clinical phenotype is a rapid increase in body temperature, tachycardia, tachypnea, and muscle rigidity. Avoidance of triggering agents is crucial. Dantrolene is a muscle relaxant that acts by interfering with",Usually autosomal-dominant RYR1 or CACNA1S susceptibility causing uncontrolled sarcoplasmic-reticulum calcium release; avoid volatile anesthetics and succinylcholine; dantrolene inhibits RyR1-mediated calcium release.
182,p07,AttentionScore (layer=17),581.610107,0.294556,0.705444,4.0,"Malignant hyperthermia (MH) is a pharmacogenetic disorder triggered by volatile anesthetics (e.g., halothane, sevoflurane) and succinylcholine, leading to uncontrolled skeletal muscle hypermetabolism, hyperthermia, and muscle rigidity. The key clinical phenotype is a rapid increase in body temperature, tachycardia, tachypnea, and muscle rigidity. Avoidance of triggering agents is crucial. Dantrolene is a muscle relaxant that acts by interfering with",Usually autosomal-dominant RYR1 or CACNA1S susceptibility causing uncontrolled sarcoplasmic-reticulum calcium release; avoid volatile anesthetics and succinylcholine; dantrolene inhibits RyR1-mediated calcium release.
336,p07,CocoaPPL,0.068765,0.543798,0.456202,5.0,"Malignant hyperthermia (MH) is a pharmacogenetic disorder triggered by volatile anesthetics (e.g., halothane, sevoflurane) and succinylcholine, leading to uncontrolled skeletal muscle hypermetabolism, hyperthermia, and muscle rigidity. The key clinical phenotype is a rapid increase in body temperature, tachycardia, tachypnea, and muscle rigidity. Avoidance of triggering agents is crucial. Dantrolene is a muscle relaxant that acts by interfering with",Usually autosomal-dominant RYR1 or CACNA1S susceptibility causing uncontrolled sarcoplasmic-reticulum calcium release; avoid volatile anesthetics and succinylcholine; dantrolene inhibits RyR1-mediated calcium release.
358,p07,CocoaMTE,0.148876,0.574754,0.425246,5.0,"Malignant hyperthermia (MH) is a pharmacogenetic disorder triggered by volatile anesthetics (e.g., halothane, sevoflurane) and succiny

In [25]:
# 2) For a specific technique, show every prompt with generated answer and score.
show_estimator_analysis("MahalanobisDistanceSeq_decoder", normalization="minmax")


,prompt_id,raw_uncertainty,normalized_confidence,normalized_uncertainty,normalized_uncertainty_rank,generated_answer,ground_truth
204,p07,10.111046,0.000000,1.000000e+00,1.0,"Malignant hyperthermia (MH) is a pharmacogenetic disorder triggered by volatile anesthetics (e.g., halothane, sevoflurane) and succinylcholine, leading to uncontrolled skeletal muscle hypermetabolism, hyperthermia, and muscle rigidity. The key clinical phenotype is a rapid increase in body temperature, tachycardia, tachypnea, and muscle rigidity. Avoidance of triggering agents is crucial. Dantrolene is a muscle relaxant that acts by interfering with",Usually autosomal-dominant RYR1 or CACNA1S susceptibility causing uncontrolled sarcoplasmic-reticulum calcium release; avoid volatile anesthetics and succinylcholine; dantrolene inhibits RyR1-mediated calcium release.
206,p09,3.014946,0.999919,8.052921e-05,2.0,"Acquired resistance to osimertinib in NSCLC primarily stems from the C797S mutation, MET amplification, and small-cell transformation. The strongest current clinical evidence supports the use of amivantamab for C797S-mutant NSCLC and MET amplification, while the treatment for small-cell transformation is less well-defined, with options including chemotherapy and immunotherapy.","Osimertinib resistance may involve EGFR C797S, MET amplification, or small-cell transformation; strategies include molecularly guided EGFR combinations, MET-targeted therapy trials/combinations, or small-cell chemotherapy regimens."
198,p01,3.014892,0.999927,7.290296e-05,3.0,"The diagnosis is ST-segment elevation myocardial infarction (STEMI), specifically an anterior wall MI. The immediate reperfusion management priority is to restore blood flow to the affected myocardium as quickly as possible, ideally within 90 minutes of first medical contact, using either percutaneous coronary intervention (PCI) or thrombolytic therapy.","Anterior STEMI involving V1-V4 with elevated troponin; activate emergent reperfusion, preferably primary PCI within guideline time targets, with antiplatelet/anticoagulant support."
205,p08,3.014836,0.999935,6.497434e-05,4.0,"Antiphospholipid syndrome (APS) is characterized by recurrent pregnancy loss, thrombosis, and positive lupus anticoagulant and anti-cardiolipin antibodies. Thrombosis in APS is caused by the formation of blood clots due to the presence of antiphospholipid antibodies, which activate the coagulation cascade. Warfarin is a traditional anticoagulant for APS, while DOACs are increasingly used, with evidence suggesting similar efficacy and safety profiles in APS patients, though further research i...","Antiphospholipid syndrome causes antibody-mediated endothelial/platelet/complement activation and thrombosis; warfarin is generally preferred over DOACs, especially in high-risk or arterial APS."
200,p03,3.014774,0.999944,5.620583e-05,5.0,"Metformin lowers blood glucose in type 2 diabetes primarily by reducing hepatic glucose production. It achieves this through AMPK activation, which decreases gluconeogenesis, and by inhibiting mitochondrial complex I, leading to reduced ATP production and increased AMP levels.","Metformin reduces hepatic glucose output mainly by inhibiting mitochondrial complex I, increasing cellular energy stress and AMPK-linked signaling, thereby suppressing gluconeogenesis."
203,p06,3.014708,0.999953,4.700057e-05,6.0,"The patient's hyperkalemia and rising creatinine after starting an ACE inhibitor are likely due to the ACE inhibitor's inhibition of aldosterone production, leading to decreased potassium excretion and worsening kidney function. Management involves discontinuing the ACE inhibitor, initiating a potassium binder (e.g., sodium polystyrene sulfonate or patiromer), and potentially adding a loop diuretic if volume overload is present.","ACE inhibition reduces angiotensin-II efferent arteriolar tone and aldosterone-mediated potassium excretion; manage hyperkalemia urgently, review ACE inhibitor/renal function, st

In [26]:
# 3) Top uncertain prompts for every technique.
show_top_uncertain_by_each_technique(normalization="minmax", top_k=3)


,estimator,prompt_id,raw_uncertainty,normalized_confidence,normalized_uncertainty,generated_answer,ground_truth
183,AttentionScore (layer=17),p08,662.857300,0.000000,1.000000,"Antiphospholipid syndrome (APS) is characterized by recurrent pregnancy loss, thrombosis, and positive lupus anticoagulant and anti-cardiolipin antibodies. Thrombosis in APS is caused by the formation of blood clots due to the presence of antiphospholipid antibodies, which activate the coagulation cascade. Warfarin is a traditional anticoagulant for APS, while DOACs are increasingly used, with evidence suggesting similar efficacy and safety profiles in APS patients, though further research i...","Antiphospholipid syndrome causes antibody-mediated endothelial/platelet/complement activation and thrombosis; warfarin is generally preferred over DOACs, especially in high-risk or arterial APS."
185,AttentionScore (layer=17),p10,614.630371,0.174844,0.825156,"XR-7291 disrupts cardiolipin remodelling in drug-resistant glioblastoma stem cells by inhibiting the enzyme cardiolipin synthase, leading to a loss of cardiolipin asymmetry and disruption of mitochondrial membrane potential. This triggers selective apoptosis in stem cells while sparing normal neural progenitors, likely due to differences in cardiolipin composition and function. The key downstream effectors are mitochondrial depolarization and caspase activation, and a potential biomarker of ...","No reliable reference answer exists because XR-7291 is fabricated. A grounded answer should explicitly state that the compound, mechanism, and biomarker cannot be verified rather than inventing details."
181,AttentionScore (layer=17),p06,594.081116,0.249343,0.750657,"The patient's hyperkalemia and rising creatinine after starting an ACE inhibitor are likely due to the ACE inhibitor's inhibition of aldosterone production, leading to decreased potassium excretion and worsening kidney function. Management involves discontinuing the ACE inhibitor, initiating a potassium binder (e.g., sodium polystyrene sulfonate or patiromer), and potentially adding a loop diuretic if volume overload is present.","ACE inhibition reduces angiotensin-II efferent arteriolar tone and aldosterone-mediated potassium excretion; manage hyperkalemia urgently, review ACE inhibitor/renal function, stop contributors, and adjust therapy."
312,CocoaMSP,p05,8.112179,0.000000,1.000000,"The next therapeutic step is to escalate to infliximab, a TNF-alpha inhibitor, due to the failure of mesalazine. The rationale is that the patient's condition is likely Crohn's disease, which is characterized by inflammation of the gastrointestinal tract and is often unresponsive to mesalazine alone.",Ulcerative colitis beyond mild disease after mesalazine failure; escalate to corticosteroids for induction and/or biologic/small-molecule therapy depending on severity and maintenance plan.
317,CocoaMSP,p10,8.065125,0.006692,0.993308,"XR-7291 disrupts cardiolipin remodelling in drug-resistant glioblastoma stem cells by inhibiting the enzyme cardiolipin synthase, leading to a loss of cardiolipin asymmetry and disruption of mitochondrial membrane potential. This triggers selective apoptosis in stem cells while sparing normal neural progenitors, likely due to differences in cardiolipin composition and function. The key downstream effectors are mitochondrial depolarization and caspase activation, and a potential biomarker of ...","No reliable reference answer exists because XR-7291 is fabricated. A grounded answer should explicitly state that the compound, mechanism, and biomarker cannot be verified rather than inventing details."
316,CocoaMSP,p09,8.040006,0.010265,0.989735,"Acquired resistance to osimertinib in NSCLC primarily stems from the C797S mutation, MET amplification, and small-cell transformation. The strongest current clinical evidence supports the use of amivantamab for C797S-mutant NSCLC and MET amplification, while the treatment for small-cell transformation is l

In [27]:
confidence_summary_df = (
    normalized_long_df
    .groupby(["prompt_id", "normalization"], as_index=False)["normalized_confidence"]
    .mean()
    .pivot(index="prompt_id", columns="normalization", values="normalized_confidence")
    .reset_index()
)
confidence_summary_df.sort_values("minmax")

normalization,prompt_id,minmax,quantile
9,p10,0.226865,0.262032
8,p09,0.329199,0.272727
4,p05,0.345962,0.342246
7,p08,0.386213,0.331551
6,p07,0.516262,0.529412
5,p06,0.628458,0.609626
3,p04,0.629152,0.598930
2,p03,0.660683,0.582888
10,p11,0.776073,0.732620
0,p01,0.902719,0.866310


In [28]:
# ============================================================
# MinMax confidence table
# ============================================================

minmax_conf_cols = [
    c for c in normalized_wide_df.columns
    if c.endswith("__minmax_confidence")
]

minmax_confidence_df = normalized_wide_df[
    ["prompt_id"] + minmax_conf_cols
].copy()

minmax_confidence_df.columns = [
    c.replace("__minmax_confidence", "") for c in minmax_confidence_df.columns
]

minmax_confidence_df

,prompt_id,MaximumSequenceProbability,Perplexity,MaximumTokenProbability,MeanTokenEntropy,TokenEntropy,SelfCertainty,PTrue,MeanPointwiseMutualInformation,AttentionScore (layer=17),MahalanobisDistanceSeq_decoder,MonteCarloSequenceEntropy,MonteCarloNormalizedSequenceEntropy,SemanticEntropy,SemanticDensity,CocoaMSP,CocoaPPL,CocoaMTE
0,p01,0.938357,1.000000,1.000000,1.000000,1.000000,0.778278,0.839140,0.602217,0.315379,0.999927,0.929977,1.000000,0.996459,1.000000,0.946494,1.000000,1.000000
1,p02,1.000000,0.789511,0.779016,0.904176,0.895401,1.000000,0.978419,1.000000,0.987948,0.999957,1.000000,0.909826,1.000000,0.528723,1.000000,0.919628,0.970080
2,p03,0.848104,0.613406,0.606008,0.588400,0.580390,0.406751,0.301532,0.164004,1.000000,0.999944,0.679799,0.364649,0.678728,0.847412,0.884911,0.836705,0.830868
3,p04,0.815698,0.497235,0.487539,0.547671,0.538286,0.422615,1.000000,0.187554,0.479692,0.999963,0.714436,0.518802,0.664301,0.885458,0.724857,0.588428,0.623045
4,p05,0.501988,0.210826,0.205027,0.204178,0.198168,0.265754,0.900734,0.203457,0.526672,0.999970,0.407840,0.184926,0.334394,0.737423,0.000000,0.000000,0.000000
5,p06,0.672159,0.683195,0.682933,0.701463,0.701319,0.476486,0.521332,0.536558,0.249343,0.999953,0.488870,0.409168,0.516065,0.954527,0.604860,0.737013,0.748548
6,p07,0.451836,0.595027,0.596766,0.637358,0.639347,0.659864,0.921001,0.954794,0.294556,0.000000,0.533350,0.666896,0.531355,0.000000,0.175753,0.543798,0.574754
7,p08,0.358860,0.413526,0.414159,0.355167,0.355719,0.393262,0.420623,0.383901,0.000000,0.999935,0.285670,0.374240,0.352354,0.530353,0.095864,0.434961,0.397029
8,p09,0.408316,0.326885,0.325461,0.325104,0.323656,0.179423,0.501345,0.404804,0.450710,0.999919,0.000000,0.000000,0.000000,0.841857,0.010265,0.247876,0.250762
9,p10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.501425,0.000000,0.174844,1.000000,0.216547,0.140800,0.132568,0.897519,0.006692,0.387258,0.399045


In [29]:
# ============================================================
# Quantile confidence table
# ============================================================

quantile_conf_cols = [
    c for c in normalized_wide_df.columns
    if c.endswith("__quantile_confidence")
]

quantile_confidence_df = normalized_wide_df[
    ["prompt_id"] + quantile_conf_cols
].copy()

quantile_confidence_df.columns = [
    c.replace("__quantile_confidence", "") for c in quantile_confidence_df.columns
]

quantile_confidence_df

,prompt_id,MaximumSequenceProbability,Perplexity,MaximumTokenProbability,MeanTokenEntropy,TokenEntropy,SelfCertainty,PTrue,MeanPointwiseMutualInformation,AttentionScore (layer=17),MahalanobisDistanceSeq_decoder,MonteCarloSequenceEntropy,MonteCarloNormalizedSequenceEntropy,SemanticEntropy,SemanticDensity,CocoaMSP,CocoaPPL,CocoaMTE
0,p01,0.909091,1.000000,1.000000,1.000000,1.000000,0.909091,0.636364,0.818182,0.454545,0.272727,0.909091,1.000000,0.909091,1.000000,0.909091,1.000000,1.000000
1,p02,1.000000,0.818182,0.818182,0.818182,0.818182,1.000000,0.909091,1.000000,0.909091,0.727273,1.000000,0.909091,1.000000,0.272727,1.000000,0.909091,0.909091
2,p03,0.727273,0.636364,0.636364,0.545455,0.545455,0.454545,0.181818,0.181818,1.000000,0.454545,0.636364,0.363636,0.727273,0.636364,0.727273,0.727273,0.727273
3,p04,0.636364,0.454545,0.454545,0.454545,0.454545,0.545455,1.000000,0.272727,0.636364,0.818182,0.727273,0.636364,0.636364,0.727273,0.636364,0.545455,0.545455
4,p05,0.454545,0.181818,0.181818,0.181818,0.181818,0.272727,0.727273,0.363636,0.727273,0.909091,0.363636,0.272727,0.272727,0.454545,0.090909,0.090909,0.090909
5,p06,0.545455,0.727273,0.727273,0.727273,0.727273,0.636364,0.545455,0.727273,0.272727,0.545455,0.454545,0.545455,0.454545,0.909091,0.545455,0.636364,0.636364
6,p07,0.363636,0.545455,0.545455,0.636364,0.636364,0.818182,0.818182,0.909091,0.363636,0.090909,0.545455,0.727273,0.545455,0.090909,0.454545,0.454545,0.454545
7,p08,0.181818,0.363636,0.363636,0.363636,0.363636,0.363636,0.272727,0.454545,0.090909,0.363636,0.272727,0.454545,0.363636,0.363636,0.363636,0.363636,0.272727
8,p09,0.272727,0.272727,0.272727,0.272727,0.272727,0.181818,0.363636,0.545455,0.545455,0.181818,0.090909,0.090909,0.090909,0.545455,0.272727,0.181818,0.181818
9,p10,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.454545,0.090909,0.181818,1.000000,0.181818,0.181818,0.181818,0.818182,0.181818,0.272727,0.363636


## 13. Ranking and correlation summaries

In [30]:
rank_matrix_df = sequence_results_df[["prompt_id"]].copy()
for col in score_cols:
    rank_matrix_df[f"{col}_rank"] = pd.to_numeric(sequence_results_df[col], errors="coerce").rank(
        method="min",
        ascending=False,
    )
rank_matrix_df

,prompt_id,MaximumSequenceProbability_rank,Perplexity_rank,MaximumTokenProbability_rank,MeanTokenEntropy_rank,TokenEntropy_rank,SelfCertainty_rank,PTrue_rank,MeanPointwiseMutualInformation_rank,AttentionScore (layer=17)_rank,MahalanobisDistanceSeq_decoder_rank,MonteCarloSequenceEntropy_rank,MonteCarloNormalizedSequenceEntropy_rank,SemanticEntropy_rank,SemanticDensity_rank,CocoaMSP_rank,CocoaPPL_rank,CocoaMTE_rank
0,p01,10.0,11.0,11.0,11.0,11.0,10.0,7.0,9.0,5.0,3.0,10.0,11.0,10.0,11.0,10.0,11.0,11.0
1,p02,11.0,9.0,9.0,9.0,9.0,11.0,10.0,11.0,10.0,8.0,11.0,10.0,11.0,3.0,11.0,10.0,10.0
2,p03,8.0,7.0,7.0,6.0,6.0,5.0,2.0,2.0,11.0,5.0,7.0,4.0,8.0,7.0,8.0,8.0,8.0
3,p04,7.0,5.0,5.0,5.0,5.0,6.0,11.0,3.0,7.0,9.0,8.0,7.0,7.0,8.0,7.0,6.0,6.0
4,p05,5.0,2.0,2.0,2.0,2.0,3.0,8.0,4.0,8.0,10.0,4.0,3.0,3.0,5.0,1.0,1.0,1.0
5,p06,6.0,8.0,8.0,8.0,8.0,7.0,6.0,8.0,3.0,6.0,5.0,6.0,5.0,10.0,6.0,7.0,7.0
6,p07,4.0,6.0,6.0,7.0,7.0,9.0,9.0,10.0,4.0,1.0,6.0,8.0,6.0,1.0,5.0,5.0,5.0
7,p08,2.0,4.0,4.0,4.0,4.0,4.0,3.0,5.0,1.0,4.0,3.0,5.0,4.0,4.0,4.0,4.0,3.0
8,p09,3.0,3.0,3.0,3.0,3.0,2.0,4.0,6.0,6.0,2.0,1.0,1.0,1.0,6.0,3.0,2.0,2.0
9,p10,1.0,1.0,1.0,1.0,1.0,1.0,5.0,1.0,2.0,11.0,2.0,2.0,2.0,9.0,2.0,3.0,4.0


In [31]:
corr_df = sequence_results_df[score_cols].apply(pd.to_numeric, errors="coerce").corr(method="spearman")
corr_df

,MaximumSequenceProbability,Perplexity,MaximumTokenProbability,MeanTokenEntropy,TokenEntropy,SelfCertainty,PTrue,MeanPointwiseMutualInformation,AttentionScore (layer=17),MahalanobisDistanceSeq_decoder,MonteCarloSequenceEntropy,MonteCarloNormalizedSequenceEntropy,SemanticEntropy,SemanticDensity,CocoaMSP,CocoaPPL,CocoaMTE
MaximumSequenceProbability,1.000000,0.854545,0.854545,0.818182,0.818182,0.790909,0.190909,0.472727,0.709091,0.027273,0.936364,0.763636,0.918182,0.009091,0.900000,0.863636,0.854545
Perplexity,0.854545,1.000000,1.000000,0.990909,0.990909,0.881818,-0.027273,0.654545,0.354545,-0.309091,0.845455,0.863636,0.881818,0.000000,0.918182,0.945455,0.918182
MaximumTokenProbability,0.854545,1.000000,1.000000,0.990909,0.990909,0.881818,-0.027273,0.654545,0.354545,-0.309091,0.845455,0.863636,0.881818,0.000000,0.918182,0.945455,0.918182
MeanTokenEntropy,0.818182,0.990909,0.990909,1.000000,1.000000,0.918182,0.036364,0.727273,0.290909,-0.345455,0.836364,0.900000,0.863636,-0.054545,0.890909,0.918182,0.890909
TokenEntropy,0.818182,0.990909,0.990909,1.000000,1.000000,0.918182,0.036364,0.727273,0.290909,-0.345455,0.836364,0.900000,0.863636,-0.054545,0.890909,0.918182,0.890909
SelfCertainty,0.790909,0.881818,0.881818,0.918182,0.918182,1.000000,0.363636,0.818182,0.290909,-0.281818,0.881818,0.954545,0.881818,-0.209091,0.845455,0.836364,0.809091
PTrue,0.190909,-0.027273,-0.027273,0.036364,0.036364,0.363636,1.000000,0.318182,0.018182,0.190909,0.309091,0.327273,0.181818,0.045455,0.081818,0.018182,0.036364
MeanPointwiseMutualInformation,0.472727,0.654545,0.654545,0.727273,0.727273,0.818182,0.318182,1.000000,0.036364,-0.490909,0.500000,0.709091,0.500000,-0.336364,0.518182,0.500000,0.463636
AttentionScore (layer=17),0.709091,0.354545,0.354545,0.290909,0.290909,0.290909,0.018182,0.036364,1.000000,0.181818,0.563636,0.200000,0.536364,-0.254545,0.490909,0.372727,0.381818
MahalanobisDistanceSeq_decoder,0.027273,-0.309091,-0.309091,-0.345455,-0.345455,-0.281818,0.190909,-0.490909,0.181818,1.000000,0.036364,-0.172727,-0.054545,0.181818,-0.154545,-0.145455,-0.081818


## 14. Save outputs

In [32]:
paths = {
    "sequence_results": OUTPUT_DIR / "lmpolygraph_sequence_results.csv",
    "native_score_matrix": OUTPUT_DIR / "lmpolygraph_native_score_matrix.csv",
    "internal_diagnostics": OUTPUT_DIR / "lmpolygraph_internal_diagnostics.csv",
    "technique_analysis_long": OUTPUT_DIR / "lmpolygraph_technique_analysis_long.csv",
    "normalized_long": OUTPUT_DIR / "lmpolygraph_normalized_long.csv",
    "normalized_wide": OUTPUT_DIR / "lmpolygraph_normalized_wide.csv",
    "confidence_summary": OUTPUT_DIR / "lmpolygraph_confidence_summary.csv",
    "sampled_generations": OUTPUT_DIR / "lmpolygraph_sampled_generations.csv",
    "reference_answers": OUTPUT_DIR / "clinical_reference_answers.csv",
    "raw_payload": OUTPUT_DIR / "lmpolygraph_raw_payload.pkl",
}

sequence_results_df.to_csv(paths["sequence_results"], index=False)
native_score_matrix_df.to_csv(paths["native_score_matrix"], index=False)
internal_diagnostics_df.to_csv(paths["internal_diagnostics"], index=False)
technique_analysis_long_df.to_csv(paths["technique_analysis_long"], index=False)
normalized_long_df.to_csv(paths["normalized_long"], index=False)
normalized_wide_df.to_csv(paths["normalized_wide"], index=False)
confidence_summary_df.to_csv(paths["confidence_summary"], index=False)
samples_df.to_csv(paths["sampled_generations"], index=False)
reference_df.to_csv(paths["reference_answers"], index=False)

with open(paths["raw_payload"], "wb") as f:
    pickle.dump({
        "manager_estimations": getattr(manager, "estimations", None),
        "manager_stats": getattr(manager, "stats", None),
        "score_cols": score_cols,
        "prompt_ids": PROMPT_IDS,
        "config": {
            "model_name": MODEL_NAME,
            "max_new_tokens": MAX_NEW_TOKENS,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "normalization_methods": NORMALIZATION_METHODS,
            "actual_samples_per_prompt": actual_samples_per_prompt,
        },
    }, f)

print("Saved outputs to:", OUTPUT_DIR.resolve())
for name, path in paths.items():
    print(f"{name}: {path}")

Saved outputs to: /content/lmpolygraph_medgemma_clinical_outputs
sequence_results: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_sequence_results.csv
native_score_matrix: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_native_score_matrix.csv
internal_diagnostics: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_internal_diagnostics.csv
technique_analysis_long: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_technique_analysis_long.csv
normalized_long: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_normalized_long.csv
normalized_wide: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_normalized_wide.csv
confidence_summary: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_confidence_summary.csv
sampled_generations: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_sampled_generations.csv
reference_answers: lmpolygraph_medgemma_clinical_outputs/clinical_reference_answers.csv
raw_payload: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_raw_payload.pkl


## 15. Notes

- `AttentionScore` uses attention tensors from the model, so eager attention is enabled during model loading.
- `MeanPointwiseMutualInformation` is the sequence-level PMI score; token-level PMI is not included to keep the table compact.
- Sampling count is reported from `sample_texts` after the run because this notebook uses LM-Polygraph's default sampling calculator configuration.